# Stock Market Analysis & Prediction using LSTM

**Stocks Analyzed:** Apple, Google, Microsoft, Amazon, Tesla, Nvidia, Meta.
**Data Source:** Yahoo Finance via yfinance
**Time Period:** Last 10 years
**Prediction Target:** Apple(AAPL) Closing Price using LSTM

## Table of Contents


## Key Questions we are asking;

1. What was the change in price of the stock over time?

2. What was the daily return of the stock on average?
3. What was the moving average of the various stocks?
4. What was the correlation between different stocks'?
5. How much value do we put at risk by investing in a particular stock?
6. How can we attempt to predict future stock behavior? (Predicting the closing price stock price of APPLE inc using LSTM)



<a id='1'></a>
## ⚙️ 1. Environment Setup

Install and import all required libraries.

In [ ]:
# Core libraries
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np 
import yfinance as yf
from datetime import datetime, timedelta

# Visualization libraries
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 12
plt.rcParams['grid.alpha'] = 0.3
sns.set_style('darkgrid')
PALETTE = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']

# Machine Learning libraries
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("✅ All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")







In [5]:
# Config

TICKERS = {
    'GOOGL': 'Google',
    'AAPL': 'Apple',
    'MSFT': 'Microsoft',
    'AMZN': 'Amazon',
    'META': 'Meta',
    'TSLA': 'Tesla',
    'NVDA': 'NVIDIA'
}

END = datetime.now()
START = datetime(END.year - 10, END.month, END.day)

# Download & tag
frames = []
for ticker, name in TICKERS.items():
    df = yf.download(ticker, start=START, end=END, progress=False, auto_adjust=True)
    df.columns = df.columns.get_level_values(0) # Flatten multi-index columns
    df['company_name'] = name.upper() # Add company name column
    frames.append(df)
    
# Combine & clean
df = pd.concat(frames, axis=0).rename(columns={'Adj Close': 'Adj Close'}).sort_index()
df.index.name = 'date'
print(df.tail(10))

Price            Close        High         Low        Open     Volume  \
date                                                                    
2026-03-19  307.130005  308.059998  302.350006  304.010010   25075800   
2026-03-19  178.559998  179.979996  175.789993  178.009995  170968500   
2026-03-19  380.299988  387.269989  378.730011  387.269989   67078300   
2026-03-20  205.369995  207.539902  204.315994  207.500000   63694603   
2026-03-20  367.959991  379.890015  364.460114  379.390015   78628603   
2026-03-20  301.000000  306.000000  298.269989  305.989990   44364079   
2026-03-20  593.659973  603.955017  587.270020  603.530029   21214898   
2026-03-20  247.990005  249.199905  246.000000  248.110001   88331081   
2026-03-20  381.850006  387.000000  380.119995  386.260010   50853154   
2026-03-20  172.929993  178.259995  171.725006  178.000000  241323528   

Price      company_name  
date                     
2026-03-19       GOOGLE  
2026-03-19       NVIDIA  
2026-03-19        T

In [6]:
df.head()

Price,Close,High,Low,Open,Volume,company_name
date,,,,,,
2016-03-23,37.567505,37.955798,37.491632,37.854633,24678000,GOOGLE
2016-03-23,24.036375,24.249267,23.984285,24.115645,102814000,APPLE
2016-03-23,0.842850,0.848970,0.828652,0.832324,429008000,NVIDIA
2016-03-23,47.487614,47.725184,47.285240,47.610798,20129000,MICROSOFT
2016-03-23,14.838667,15.648667,14.802000,15.491333,74232000,TESLA
